In [16]:
# -*- coding: utf-8 -*-
import re
import numpy as np
import pandas as pd
from sklearn.model_selection import KFold
from sklearn.cluster import KMeans

# =========================
# —— KMeans 城市中心距离——
# =========================
_R_EARTH_KM = 6371.0088

def _haversine_km(lon1, lat1, lon2, lat2):
    lon1, lat1, lon2, lat2 = map(np.radians, [lon1, lat1, lon2, lat2])
    dlon, dlat = lon2 - lon1, lat2 - lat1
    a = np.sin(dlat/2.0)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon/2.0)**2
    return 2.0 * _R_EARTH_KM * np.arcsin(np.sqrt(a))

def _kmeans_city_centers_one(df_city, k, lon_col, lat_col, random_state=42):
    """对单个城市的 (lon,lat) 做 KMeans(k)，返回各样本到最近中心的距离（km），索引对齐 df_city.index。"""
    idx = df_city.index
    g = df_city[[lon_col, lat_col]].dropna()
    # 若无有效坐标或样本少于 k，直接返回 NaN
    if g.shape[0] < max(1, k):
        out = pd.Series(np.nan, index=idx, name="dist_to_city_center_km")
        return out

    lat0 = float(g[lat_col].median())
    X = np.c_[ g[lon_col].values * np.cos(np.radians(lat0)), g[lat_col].values ]

    km = KMeans(n_clusters=k, n_init=10, random_state=random_state)
    labels = km.fit_predict(X)
    Cx, Cy = km.cluster_centers_[:, 0], km.cluster_centers_[:, 1]
    # 反变换中心为经纬度
    lon_c = Cx / np.cos(np.radians(lat0))
    lat_c = Cy

    # 到每个中心的球面距离，取最小
    lon_arr, lat_arr = g[lon_col].values, g[lat_col].values
    dmat = np.vstack([
        _haversine_km(lon_arr, lat_arr, lon_c[i], lat_c[i])
        for i in range(k)
    ]).T
    dist_min = dmat.min(axis=1)

    out = pd.Series(np.nan, index=idx, name="dist_to_city_center_km")
    out.loc[g.index] = dist_min
    return out

def _add_city_center_distance_kmeans(df, city_col="城市", lon_col="lon", lat_col="lat", k_per_city=1, random_state=42):
    """
    仅生成一列 dist_to_city_center_km：
    - 对每个城市分别在 (lon,lat) 上做 KMeans(k)，k 可为 int 或 {城市名:k} 的 dict；
    - 计算每条样本到最近中心的 Haversine 距离（km）。
    """
    # 兼容 lat 命名为 'lot' 的情况
    if lat_col not in df.columns and "lot" in df.columns:
        lat_col = "lot"

    if (city_col not in df.columns) or (lon_col not in df.columns) or (lat_col not in df.columns):
        # 缺关键列则直接返回 NaN 列
        return pd.Series(np.nan, index=df.index, name="dist_to_city_center_km")

    # 解析 k_per_city
    if isinstance(k_per_city, int):
        k_default, k_map = k_per_city, {}
    elif isinstance(k_per_city, dict):
        k_default, k_map = 1, k_per_city.copy()
    else:
        raise ValueError("k_per_city 必须是 int 或 dict")

    pieces = []
    for city, g in df.groupby(city_col, sort=False):
        k = k_map.get(city, k_default)
        k = max(1, int(k))
        dist_s = _kmeans_city_centers_one(g, k, lon_col, lat_col, random_state=random_state)
        pieces.append(dist_s)

    dist_all = pd.concat(pieces).loc[df.index]
    dist_all.name = "dist_to_city_center_km"
    return dist_all


# =========================
# —— 基础小工具 —— 
# =========================
def _to_float_first(x):
    """从字符串里抓第一个数字（含千分位/小数）→ float；抓不到返回 NaN"""
    if pd.isna(x): return np.nan
    s = str(x)
    m = re.search(r"(\d{1,3}(?:,\d{3})+|\d+)(?:\.(\d+))?", s)
    if not m: return np.nan
    whole = m.group(1).replace(",", "")
    frac = m.group(2) or ""
    return float(whole + (("." + frac) if frac else ""))

def _extract_all_numbers(s):
    """提取所有数字（含千分位/小数）→ [floats]"""
    if pd.isna(s): return []
    s = str(s)
    out = []
    for m in re.findall(r"(\d{1,3}(?:,\d{3})+|\d+)(?:\.(\d+))?", s):
        whole = m[0].replace(",", "")
        frac = m[1]
        out.append(float(whole + ("." + frac if frac else "")))
    return out

def _parse_year_only(x):
    """提取 4 位年份（如 2019）"""
    if pd.isna(x): return np.nan
    m = re.search(r"(\d{4})", str(x))
    return float(m.group(1)) if m else np.nan

def _process_building_year(year_str):
    """处理‘建筑年代’列 → (起始年, 结束年)；若只有一个年，起止相同"""
    if pd.isna(year_str) or str(year_str) == '未知': 
        return np.nan, np.nan
    try:
        years = re.findall(r'\d{4}', str(year_str))
        if len(years) >= 2: return int(years[0]), int(years[1])
        elif len(years) == 1: return int(years[0]), int(years[0])
        else: return np.nan, np.nan
    except:
        return np.nan, np.nan

def _parse_layout_4(x):
    """从‘房屋户型’里抽‘室/厅/卫/厨’数量"""
    if pd.isna(x): 
        return (np.nan, np.nan, np.nan, np.nan)
    s = str(x)
    room = re.search(r"(\d+)\s*(?:室|房)", s)
    hall = re.search(r"(\d+)\s*(?:厅|客厅)", s)
    bath = re.search(r"(\d+)\s*(?:卫|卫生间|厕)", s)
    kitc = re.search(r"(\d+)\s*(?:厨|厨房)", s)
    R = int(room.group(1)) if room else np.nan
    H = int(hall.group(1)) if hall else np.nan
    B = int(bath.group(1)) if bath else np.nan
    K = int(kitc.group(1)) if kitc else np.nan
    return (R,H,B,K)

def _parse_floor(floor_text):
    """
    所在楼层：识别‘共x层’→ 总楼层；对‘地下室/低楼层/底层/中楼层/高楼层/顶层’做次序编码。
    返回：(floor_level_code, total_floors)
    """
    if pd.isna(floor_text): return (np.nan, np.nan)
    s = str(floor_text)

    # 总楼层
    tot = np.nan
    m_tot = re.search(r"共\s*(\d+)\s*层", s)
    if m_tot: tot = float(m_tot.group(1))

    # 次序编码
    lvl = np.nan
    if re.search(r"地下室", s): lvl = -1
    elif re.search(r"(低楼层|底层)", s): lvl = 1
    elif re.search(r"中楼层", s): lvl = 2
    elif re.search(r"(高楼层|顶层)", s): lvl = 3
    else:
        m_cur = re.search(r"(\d+)\s*层", s)
        if m_cur and not np.isnan(tot):
            cur = float(m_cur.group(1))
            if cur <= max(1.0, tot * 1/3): lvl = 1
            elif cur <= max(2.0, tot * 2/3): lvl = 2
            else: lvl = 3
    return (lvl, tot)

_CN_NUM = {"零":0,"〇":0,"一":1,"二":2,"两":2,"三":3,"四":4,"五":5,"六":6,"七":7,"八":8,"九":9,"十":10}
def _cn_digits_to_int(s):
    """
    简化版中文小数字 → int，例如：'两'→2，'十'→10，'十二'→12，'二十'→20，'二十三'→23。
    仅覆盖‘梯户比例’常见写法，复杂大数可按需扩展。
    """
    if s is None: return np.nan
    s = str(s).strip()
    # 含阿拉伯数字直接取第一个
    m = re.search(r"\d+", s)
    if m: return int(m.group(0))
    # 纯中文数字（<=99）
    if not s: return np.nan
    total = 0
    if "十" in s:
        parts = s.split("十")
        left = _CN_NUM.get(parts[0], 1 if parts[0]=="" else np.nan)  # ""十X → 1十X
        right = _CN_NUM.get(parts[1], 0 if parts[1]=="" else np.nan) if len(parts)>1 else 0
        if np.isnan(left) or np.isnan(right): return np.nan
        total = int(left)*10 + int(right)
    else:
        # 单个中文数
        v = _CN_NUM.get(s, np.nan)
        total = int(v) if not pd.isna(v) else np.nan
    return total

def _parse_ti_hu_ratio(x):
    """
    ‘梯户比例’：抓 ‘X梯Y户’（X/Y 可中文或数字），输出 Y/X 的户/梯比；缺失返回 NaN。
    """
    if pd.isna(x): return np.nan
    s = str(x)
    m = re.search(r"([一二两三四五六七八九十\d]+)\s*梯\s*([一二两三四五六七八九十\d]+)\s*户", s)
    if not m: return np.nan
    ti  = _cn_digits_to_int(m.group(1))
    hu  = _cn_digits_to_int(m.group(2))
    if pd.isna(ti) or pd.isna(hu) or ti == 0: return np.nan
    return float(hu) / float(ti)

def _parse_parking_fee(x):
    """停车费：优先抓‘元’前面的数；若多处出现，取均值；抓不到返回 NaN"""
    if pd.isna(x): return np.nan
    s = str(x)
    nums_with_yuan = []
    for m in re.finditer(r"((\d{1,3}(?:,\d{3})+|\d+)(?:\.(\d+))?)\s*元", s):
        num = m.group(1).replace(",", "")
        nums_with_yuan.append(float(num))
    if nums_with_yuan:
        return float(np.mean(nums_with_yuan))
    nums = _extract_all_numbers(s)
    return float(np.mean(nums)) if nums else np.nan

def _sentiment_triplet(text):
    """客户反馈的简单情感：(正计数, 负计数, 情感分)"""
    pos = ["满意","干净","安静","方便","舒适","通透","采光好","靠谱","宽敞","安全","便利","景观","通风"]
    neg = ["脏","吵","不方便","拥堵","贵","潮","霉","不安全","差评","噪音","漏水","太小","异味"]
    if pd.isna(text): return (0,0,0.0)
    s = str(text)
    p = sum(w in s for w in pos)
    n = sum(w in s for w in neg)
    score = (p-n)/(p+n) if (p+n)>0 else 0.0
    return (p,n,score)

def _map_ring(v):
    """环线位置 → 有序数值"""
    if pd.isna(v): return 8
    s = str(v)
    if "内环内" in s: return 1
    if "一至二环" in s: return 2
    if "二至三环" in s: return 3
    if "三至四环" in s: return 4
    if "四至五环" in s: return 5
    if "五至六环" in s: return 6
    if "外环" in s or "六环" in s: return 7
    return 8

# —— 文本四列“单分数”方案：每列只输出一个数值（不爆列）——
def _z(series):
    x = pd.to_numeric(series, errors="coerce")
    mu, sd = x.mean(), x.std(ddof=0)
    sd = sd if sd and sd>0 else 1.0
    return (x - mu) / sd

def _single_text_scores(df, col_core, col_layout, col_facil, col_trans):
    """
    每个文本列仅产出 1 个分数：core_score/layout_score/facil_score/trans_score
    权重可按需调整。
    """
    out = pd.DataFrame(index=df.index)
    # 核心卖点
    if col_core in df.columns:
        s = df[col_core].fillna("")
        hot_kw  = ["学区","地铁口","南北通透","满五唯一","精装","次新","景观","采光","公园","商圈","一梯两户"]
        hype_kw = ["绝佳","稀缺","必看","抢手","拎包入住","低于市场","性价比高"]
        def _len_digit(txt):
            if not isinstance(txt, str): return (0,0.0)
            t = txt.strip()
            if not t: return (0,0.0)
            L = len(t); digit = sum(ch.isdigit() for ch in t)/L
            return (L, digit)
        core_len   = _z([_len_digit(t)[0] for t in s])
        core_digit = _z([_len_digit(t)[1] for t in s])
        core_hot   = _z([sum(len(re.findall(re.escape(w), t)) for w in hot_kw)  for t in s])
        core_hype  = _z([sum(len(re.findall(re.escape(w), t)) for w in hype_kw) for t in s])
        # 简易情感
        def _sent(txt):
            pos = ["满意","干净","安静","方便","舒适","通透","采光好","靠谱","宽敞","安全","便利","景观","通风"]
            neg = ["脏","吵","不方便","拥堵","贵","潮","霉","不安全","差评","噪音","漏水","太小","异味"]
            p = sum(w in txt for w in pos)
            n = sum(w in txt for w in neg)
            return (p-n)/(p+n) if (p+n)>0 else 0.0
        core_sent  = _z([_sent(t) for t in s])
        out["core_score"] = 0.5*core_len + 1.5*core_hot + 0.5*core_hype + 1.0*core_sent + 0.3*core_digit

    # 户型介绍
    if col_layout in df.columns:
        s = df[col_layout].fillna("")
        good_kw = ["南北通透","明厨明卫","双阳台","动静分区","主卧带卫","独立衣帽间","飘窗","采光","通风"]
        func_kw = ["书房","储物间","步入式衣帽间"]
        def _len_digit(txt):
            if not isinstance(txt, str): return (0,0.0)
            t = txt.strip()
            if not t: return (0,0.0)
            L = len(t); digit = sum(ch.isdigit() for ch in t)/L
            return (L, digit)
        layout_len   = _z([_len_digit(t)[0] for t in s])
        layout_digit = _z([_len_digit(t)[1] for t in s])
        layout_good  = _z([sum(len(re.findall(re.escape(w), t)) for w in good_kw) for t in s])
        layout_func  = _z([sum(len(re.findall(re.escape(w), t)) for w in func_kw) for t in s])
        layout_bal   = _z([len(re.findall("阳台", t)) for t in s])
        out["layout_score"] = 0.3*layout_len + 1.2*layout_good + 0.8*layout_func + 0.6*layout_bal + 0.2*layout_digit

    # 周边配套
    if col_facil in df.columns:
        s = df[col_facil].fillna("")  # 处理空值，填充为空字符串
        cat = {
            "school": ["学校", "小学", "中学", "幼儿园", "大学", "学区"],
            "hospital": ["医院", "三甲", "门诊", "卫生服务站"],
            "mall": ["商场", "购物中心", "商业街", "步行街", "超市", "菜市场", "市集"],
            "park": ["公园", "绿地", "景区", "河景", "湖景"],
            "office": ["写字楼", "产业园", "商务区", "CBD"],
            "sport": ["体育馆", "球场", "健身房", "游泳馆"],
            "noise": ["高架", "立交", "铁路", "轨道", "机场"],
        }
    
        def _count(txt, kws):
            return sum(len(re.findall(re.escape(w), txt)) for w in kws)
    
        fac_scores = {k: _z([_count(t, v) for t in s]) for k, v in cat.items()}
    
        # 计算文本长度与数字占比
        def _len_digit(txt):
            if not isinstance(txt, str): return (0, 0.0)
            t = txt.strip()
            if not t: return (0, 0.0)
            L = len(t)
            digit = sum(ch.isdigit() for ch in t) / L
            return (L, digit)
    
        fac_len = _z([_len_digit(t)[0] for t in s])
        fac_digit = _z([_len_digit(t)[1] for t in s])
    
        # 合并计算“周边配套”分数
        out["facil_score"] = (
            1.0 * fac_scores["school"] + 0.8 * fac_scores["hospital"] + 0.7 * fac_scores["mall"] +
            0.8 * fac_scores["park"] + 0.5 * fac_scores["office"] + 0.4 * fac_scores["sport"] +
            (-0.8) * fac_scores["noise"] + 0.2 * fac_len + 0.2 * fac_digit
        )
    

       # 交通出行（去掉距离抽取，避免 NaN）
    if col_trans in df.columns:
        s = df[col_trans].fillna("").astype(str)

        # 关键词计数
        metro_keywords = ["地铁", "轻轨", "地铁站", "地铁口"]
        bus_keywords   = ["公交", "公交站", "公交车", "公交线路"]
        walk_keywords  = ["步行", "走路", "步行距离"]

        def _count_kw(text, kws):
            # 返回包含的关键词个数（空文本则 0）
            return sum(k in text for k in kws)

        # 号线识别：统计出现的“X号线”的去重数量
        line_pat = r'(\d+)\s*号线'

        lines_raw   = [len({m.group(1) for m in re.finditer(line_pat, t)}) for t in s]
        metro_raw   = [_count_kw(t, metro_keywords) for t in s]
        bus_raw     = [_count_kw(t, bus_keywords)   for t in s]
        walk_raw    = [_count_kw(t, walk_keywords)  for t in s]
        station_raw = [sum(w in t for w in ["地铁","站","换乘","轻轨","城铁"]) for t in s]

        # 文本长度与数字占比（空文本→0）
        def _len_digit(txt):
            t = txt.strip()
            if not t: return (0, 0.0)
            L = len(t)
            digit = sum(ch.isdigit() for ch in t) / L
            return (L, digit)

        tr_len_raw   = [_len_digit(t)[0] for t in s]
        tr_digit_raw = [_len_digit(t)[1] for t in s]

        # z 标准化（_z 会自动避免除以 0；输入均为确定数，不含 NaN）
        lines   = _z(lines_raw)
        metro   = _z(metro_raw)
        bus     = _z(bus_raw)
        walk    = _z(walk_raw)
        station = _z(station_raw)
        tr_len  = _z(tr_len_raw)
        tr_digit= _z(tr_digit_raw)

        # 交通得分（无距离项，不会产生 NaN）
        out["trans_score"] = (
            1.2*lines + 1.0*metro + 0.8*bus + 0.5*walk + 0.8*station + 0.2*tr_len + 0.1*tr_digit
        )



    # 转数值
    for c in out.columns:
        out[c] = pd.to_numeric(out[c], errors="coerce")
    return out


# =========================
# —— 主函数：一键处理（集成 KMeans 距离 + 删除 lon/lot）—— 
# =========================
def process_real_estate_file(
    input_path: str,
    output_path: str = "processed_numeric.csv",
    core_col="核心卖点", layout_col="户型介绍", facil_col="周边配套", trans_col="交通出行",
    ring_col="环线位置",
    drop_exact_cols=("环线","开发商","物业公司","物业办公电话"),
    catboost_cols=("交易权属","房屋用途","房屋优势","物业类别","产权描述"),
    onehot_cols=("建筑结构","别墅类别","房屋年限","产权所属",
                 "装修情况","配备电梯","车位","用水","用电","燃气","采暖","供水","供暖","供电"),
    year_col="建筑年代",
    floor_col="所在楼层",
    layout_all_col="房屋户型",
    ratio_col="梯户比例",
    feedback_col="客户反馈",
    area_cols=("建筑面积","套内面积"),
    unit_strip_cols=("房屋总数","楼栋总数","绿化率","物业费","燃气费","供暖费"),
    time_cols=("交易时间","上次交易"),
    # —— 新增经纬度/城市参数 —— #
    city_col="城市", lon_col="lon", lat_col="lat",
    k_per_city=1,                       # 可为 int 或 dict，例如 {"北京市":3,"上海市":3}
    heading_print=True
):
    """
    读取 input_path，生成数值特征并新增一列：dist_to_city_center_km（KMeans-城市中心距离，单位：km）。
    然后删除原始的 lon 与 lot 列，其余逻辑保持不变。
    """
    df = pd.read_csv(input_path)
    df_feat = pd.DataFrame(index=df.index)

    # 0) 识别并保留 ID
    id_col = None
    for c in ["ID","Id","id"]:
        if c in df.columns:
            id_col = c; break
    if id_col:
        df_feat[id_col] = df[id_col]
    if heading_print:
        print(f"[INFO] Loaded: {df.shape}  | ID_col={id_col}")

    # ---- 新增：仅生成城市中心距离列 ----
    df_feat["dist_to_city_center_km"] = _add_city_center_distance_kmeans(
        df, city_col=city_col, lon_col=lon_col, lat_col=lat_col, k_per_city=k_per_city, random_state=42
    )

    # ---- 仅删除原始经纬度列（lon 与 lot）与重复列其余不变 ----
    for _col in ["lon", "lat","coord_x","coord_y","板块"]:
        if _col in df.columns:
            df = df.drop(columns=[_col])

    # A) 删除指定列（注意：不要误删‘环线位置’）
    for c in drop_exact_cols:
        if c in df.columns and c != ring_col:
            if heading_print: print(f"[DROP] {c}")
            df = df.drop(columns=[c])

    # B) 环线位置 → 有序数值 ring_num
    if ring_col in df.columns:
        df_feat["ring_num"] = df[ring_col].apply(_map_ring).astype(float)

    # D) One-Hot（drop_first）
    for c in onehot_cols:
        if c in df.columns:
            dmy = pd.get_dummies(df[c].astype(str), prefix=c, drop_first=True, dummy_na=False)
            df_feat = pd.concat([df_feat, dmy.astype(float)], axis=1)

    # E) 房屋户型 → 室/厅/卫/厨
    if layout_all_col in df.columns:
        quad = df[layout_all_col].apply(_parse_layout_4)
        df_feat["室数量"] = quad.apply(lambda t: t[0])
        df_feat["厅数量"] = quad.apply(lambda t: t[1])
        df_feat["卫数量"] = quad.apply(lambda t: t[2])
        df_feat["厨数量"] = quad.apply(lambda t: t[3])

    # F) 楼层
    if floor_col in df.columns:
        pair = df[floor_col].apply(_parse_floor)
        df_feat["楼层次序编码"] = pair.apply(lambda t: t[0])
        df_feat["总楼层"] = pair.apply(lambda t: t[1])

    # G) 梯户比
    if ratio_col in df.columns:
        df_feat["梯户比"] = df[ratio_col].apply(_parse_ti_hu_ratio)

    # H) 文本四列分数
    df_scores = _single_text_scores(df, core_col, layout_col, facil_col, trans_col)
    df_feat = df_feat.join(df_scores)

    # I) 建筑年代
    if year_col in df.columns:
        years = df[year_col].apply(_process_building_year)
        df_feat["建筑起始年份"] = years.apply(lambda x: x[0])
        df_feat["建筑结束年份"] = years.apply(lambda x: x[1])
        df_feat["建筑年限"] = 2025 - df_feat["建筑结束年份"]

    # J) 去单位 + 面积
    for c in list(unit_strip_cols) + list(area_cols):
        if c in df.columns:
            s = df[c].astype(str)
            if c == "绿化率":
                pct = pd.to_numeric(s.str.extract(r"([\d\.]+)\s*%")[0], errors="coerce")
                raw = s.apply(_to_float_first)
                df_feat[c] = np.where(~pct.isna(), pct/100.0, raw)
            else:
                df_feat[c] = s.apply(_to_float_first)

    # K) 停车费
    if "停车费" in df.columns:
        df_feat["停车费"] = df["停车费"].apply(_parse_parking_fee)

    # L) 客户反馈情感
    if feedback_col in df.columns:
        senti = df[feedback_col].apply(_sentiment_triplet)
        df_feat["feedback_pos_cnt"] = senti.apply(lambda t: t[0]).astype(float)
        df_feat["feedback_neg_cnt"] = senti.apply(lambda t: t[1]).astype(float)
        df_feat["feedback_score"]   = senti.apply(lambda t: t[2]).astype(float)

    # M) 朝向编码
    if "房屋朝向" in df.columns:
        mp = {"东":1,"南":2,"西":3,"北":4}
        df_feat["朝向编码"] = df["房屋朝向"].apply(lambda x: mp.get(str(x).strip()[:1], 0) if pd.notna(x) else 0).astype(float)

    # N) 交易年份/上次交易 & 间隔
    trade_year = None
    if "交易时间" in df.columns:
        df_feat["交易年份"] = df["交易时间"].apply(_parse_year_only)
        trade_year = df_feat["交易年份"]
    if "上次交易" in df.columns:
        df_feat["上次交易年份"] = df["上次交易"].apply(_parse_year_only)
        if trade_year is None:
            trade_year = df_feat["上次交易年份"]
    if ("交易年份" in df_feat.columns) and ("上次交易年份" in df_feat.columns):
        df_feat["交易间隔_年"] = df_feat["交易年份"] - df_feat["上次交易年份"]

    # O) 原表中可直接转数值的列
    for c in df.columns:
        if c in df_feat.columns: 
            continue
        if c in drop_exact_cols and c != ring_col:
            continue
        ser = pd.to_numeric(df[c], errors="coerce")
        if ser.notna().any():
            df_feat[c] = ser

    # P) 仅保留数值 & ID（若有）
    for col in list(df_feat.columns):
        df_feat[col] = pd.to_numeric(df_feat[col], errors="coerce")
    keep_cols = [id_col] + [c for c in df_feat.columns if c != id_col] if id_col else list(df_feat.columns)
    df_out = df_feat[keep_cols]

    # Q) 写出
    if output_path:
        df_out.to_csv(output_path, index=False)
        if heading_print:
            print(f"[SAVE] {output_path}  shape={df_out.shape}")

    return df_out




In [17]:
out = process_real_estate_file(
     input_path="ruc_Class25Q2_train_price.csv",
     output_path="ruc_Class25Q2_train_price__processed.csv",
     city_col="城市", lon_col="lon", lat_col="lat",
     k_per_city={}  # 其他城市默认 k=1
 )
# out.head()

/var/folders/q_/f08b1_dd1kd_lmlng6mq4cc00000gn/T/ipykernel_92000/3502081326.py:423: DtypeWarning: Columns (3,32,34,43,46,49,51) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(input_path)


[INFO] Loaded: (103871, 55)  | ID_col=None
[DROP] 环线
[DROP] 开发商
[DROP] 物业公司
[DROP] 物业办公电话
[SAVE] ruc_Class25Q2_train_price__processed.csv  shape=(103871, 66)


In [11]:
out.head()

,dist_to_city_center_km,ring_num,建筑结构_未知结构,建筑结构_框架结构,建筑结构_混合结构,建筑结构_砖木结构,建筑结构_砖混结构,建筑结构_钢混结构,建筑结构_钢结构,房屋年限_未满两年,...,城市,区域,板块,Price,年份,区县,板块_comm,容 积 率,停车位,停车费用
0,4.740485,3.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,...,0,109.0,150.0,6.194049e+06,2018.0,109.0,150.0,3.00,300.0,NaN
1,17.194796,6.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,...,0,65.0,299.0,4.354153e+06,2017.0,65.0,299.0,1.73,1550.0,150.0
2,31.623999,6.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,...,0,62.0,911.0,3.321992e+06,2018.0,62.0,911.0,1.70,324.0,150.0
3,40.037766,7.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,...,0,123.0,1102.0,7.895656e+06,2020.0,123.0,1102.0,1.00,500.0,NaN
4,11.184400,4.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,...,0,81.0,295.0,1.902960e+06,2017.0,81.0,295.0,1.58,1800.0,1200.0


In [26]:
# -*- coding: utf-8 -*-
# 目的：y 做 log1p 训练，评估还原到原始尺度；放宽异常值；最小特征工程（面积_log1p、area×ring）
# 修复：Imputer 保留全空列 + 统一填 0；CV 使用 sklearn.base.clone

import os, re, warnings, numpy as np, pandas as pd, joblib
from sklearn.model_selection import train_test_split, GridSearchCV, KFold
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression, Lasso, Ridge, ElasticNet
from sklearn.metrics import mean_absolute_error
from sklearn.base import clone

warnings.filterwarnings("ignore")

DATA_PATH = "ruc_Class25Q2_train_price__processed.csv"
RANDOM_STATE = 111
TEST_SIZE = 0.2
N_CV = 6
MODEL_DIR = "./models"

TARGET_KEYS = ["Price","price","租金","总价","成交价","月租","租价","y","target","label"]
LEAK_PATTERNS = [r"价格", r"单价", r"均价", r"总价", r"成交价", r"租金", r"租价", r"Price", r"price", r"每平米"]

def rmae(y_true, y_pred, eps=1e-8):
    y_true = np.asarray(y_true); y_pred = np.asarray(y_pred)
    return np.mean(np.abs(y_pred - y_true) / np.maximum(np.abs(y_true), eps))

# 1) 读取 + 目标列识别
df = pd.read_csv(DATA_PATH)
target_col = None
for k in TARGET_KEYS:
    if k in df.columns and pd.to_numeric(df[k], errors="coerce").notna().mean() >= 0.5:
        target_col = k; break
if target_col is None:
    numeric_cols = [c for c in df.columns if pd.api.types.is_numeric_dtype(df[c])]
    target_col = max(numeric_cols, key=lambda c: df[c].notna().sum())

y_full = pd.to_numeric(df[target_col], errors="coerce")
X_full = df.drop(columns=[target_col])

# 2) 去泄露（按列名）+ 仅数值列
def is_leaky(colname): return any(re.search(p, colname, flags=re.I) for p in LEAK_PATTERNS)
leak_cols = [c for c in X_full.columns if is_leaky(c)]
if leak_cols:
    print("[WARN] drop leakage-like columns:", leak_cols)
    X_full = X_full.drop(columns=leak_cols)
X_full = X_full[[c for c in X_full.columns if pd.api.types.is_numeric_dtype(X_full[c])]]

# 3) 划分（80/20）
mask = y_full.notna()
X = X_full.loc[mask].copy(); y = y_full.loc[mask].copy()
X_train, X_test, y_train_orig, y_test_orig = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE
)

print("[INFO] split shapes:", X_train.shape, X_test.shape)

# 4) 缺失填充（仅训练集统计）——关键修复：保留全空列，并在之后把全空列置 0
imp = SimpleImputer(strategy="median", keep_empty_features=True)
X_train_imp_np = imp.fit_transform(X_train)
X_test_imp_np  = imp.transform(X_test)

X_train_imp = pd.DataFrame(X_train_imp_np, columns=X_train.columns, index=X_train.index)
X_test_imp  = pd.DataFrame(X_test_imp_np,  columns=X_test.columns,  index=X_test.index)

empty_cols = [c for c in X_train.columns if X_train[c].isna().all()]
if empty_cols:
    print("[WARN] all-NaN columns filled with 0:", empty_cols[:20], "..." if len(empty_cols)>20 else "")
    X_train_imp[empty_cols] = X_train_imp[empty_cols].fillna(0.0)
    X_test_imp[empty_cols]  = X_test_imp[empty_cols].fillna(0.0)

# 5) 放宽异常值规则（仅训练集）
Q1, Q3 = X_train_imp.quantile(0.25), X_train_imp.quantile(0.75)
IQR = Q3 - Q1
lower = Q1 - 2.0 * IQR
upper = Q3 + 2.0 * IQR
keep_iqr_relaxed = ((X_train_imp >= lower) & (X_train_imp <= upper)).all(axis=1)

mu = X_train_imp.mean()
sd = X_train_imp.std(ddof=0).replace(0, 1.0)
Z = (X_train_imp - mu) / sd
keep_z_relaxed = (Z.abs() <= 4.0).all(axis=1)

keep_mask = keep_iqr_relaxed | keep_z_relaxed
X_train_clean = X_train_imp.loc[keep_mask].copy()
y_train_orig = y_train_orig.loc[keep_mask].copy()

print(f"[INFO] train kept={len(X_train_clean)}  test={len(X_test_imp)}")

# 6) 最小特征工程：面积_log1p、area×ring
def add_minimal_fe(Xtr: pd.DataFrame, Xte: pd.DataFrame):
    Xtr2, Xte2 = Xtr.copy(), Xte.copy()
    if "面积" in Xtr2.columns:
        Xtr2["面积_log1p"] = np.log1p(Xtr2["面积"].clip(lower=0))
        Xte2["面积_log1p"] = np.log1p(Xte2["面积"].clip(lower=0))
    if {"面积","ring_num"}.issubset(Xtr2.columns):
        Xtr2["area_x_ring"] = Xtr2["面积"] * Xtr2["ring_num"]
        Xte2["area_x_ring"] = Xte2["面积"] * Xte2["ring_num"]
    return Xtr2, Xte2

X_train_fe, X_test_fe = add_minimal_fe(X_train_clean, X_test_imp)

# 7) y 做 log1p 变换（训练域）
y_train_log = np.log1p(y_train_orig.clip(lower=0))
y_test_log  = np.log1p(y_test_orig.clip(lower=0))  # 仅用于观察

# 8) 轻量特征选择（去弱相关、去高相关、简易VIF）
corr = pd.DataFrame({"corr": X_train_fe.corrwith(y_train_log)})
keep_corr = corr.index[(corr["corr"].abs() >= 0.02)].tolist()
if len(keep_corr) == 0:
    # 兜底：保留前 20 个方差最大的列，避免空特征集
    keep_corr = X_train_fe.var().sort_values(ascending=False).index[:20].tolist()

X_train_fs = X_train_fe[keep_corr].copy(); X_test_fs = X_test_fe[keep_corr].copy()

corr_mat = X_train_fs.corr().abs()
upper = corr_mat.where(np.triu(np.ones(corr_mat.shape), k=1).astype(bool))
high_cols = [c for c in upper.columns if any(upper[c] > 0.8)]
if high_cols:
    X_train_fs.drop(columns=high_cols, inplace=True, errors="ignore")
    X_test_fs.drop(columns=high_cols, inplace=True, errors="ignore")

def compute_vif(Xm: pd.DataFrame):
    from sklearn.linear_model import LinearRegression
    vifs = {}
    for col in Xm.columns:
        y_i = Xm[col].values
        X_i = Xm.drop(columns=[col]).values
        if X_i.shape[1]==0: vifs[col]=1.0; continue
        r2 = LinearRegression().fit(X_i, y_i).score(X_i, y_i)
        vifs[col] = np.inf if r2>=0.9999 else 1/(1-r2)
    return pd.Series(vifs)

while True:
    if X_train_fs.shape[1] <= 5: break
    vif = compute_vif(X_train_fs.fillna(0))
    mx = vif.idxmax(); mv = float(vif.max())
    if mv > 10 and len(X_train_fs.columns) > 5:
        X_train_fs.drop(columns=[mx], inplace=True); X_test_fs.drop(columns=[mx], inplace=True)
    else:
        break

print("[INFO] final #features:", X_train_fs.shape[1])

# 9) 四个线性模型（在 y_log 上训练），并调参
models = {}

pipe_ols = Pipeline([("sc", StandardScaler()), ("lr", LinearRegression())]).fit(X_train_fs, y_train_log)
models["OLS"] = pipe_ols

pipe_lasso = Pipeline([("sc", StandardScaler()), ("ls", Lasso(max_iter=10000, random_state=RANDOM_STATE))])
gs_lasso = GridSearchCV(pipe_lasso, {"ls__alpha": np.logspace(-3, 1, 12)}, cv=N_CV,
                        scoring="neg_mean_absolute_error", n_jobs=-1)
gs_lasso.fit(X_train_fs, y_train_log); models["Lasso"] = gs_lasso.best_estimator_
print(f"[Tune] Lasso alpha={gs_lasso.best_params_['ls__alpha']}")

pipe_ridge = Pipeline([("sc", StandardScaler()), ("rg", Ridge(random_state=RANDOM_STATE))])
gs_ridge = GridSearchCV(pipe_ridge, {"rg__alpha": np.logspace(-3, 3, 13)}, cv=N_CV,
                        scoring="neg_mean_absolute_error", n_jobs=-1)
gs_ridge.fit(X_train_fs, y_train_log); models["Ridge"] = gs_ridge.best_estimator_
print(f"[Tune] Ridge alpha={gs_ridge.best_params_['rg__alpha']}")

pipe_en = Pipeline([("sc", StandardScaler()), ("en", ElasticNet(max_iter=10000, random_state=RANDOM_STATE))])
param_grid_en = {"en__alpha": np.logspace(-3, 1, 8), "en__l1_ratio": np.linspace(0.05, 0.95, 10)}
gs_en = GridSearchCV(pipe_en, param_grid_en, cv=N_CV,
                     scoring="neg_mean_absolute_error", n_jobs=-1)
gs_en.fit(X_train_fs, y_train_log); models["ElasticNet"] = gs_en.best_estimator_
print(f"[Tune] ElasticNet alpha={gs_en.best_params_['en__alpha']}, l1_ratio={gs_en.best_params_['en__l1_ratio']}")

# 10) 评估（还原到原始 y 空间）
def report_model(name, model, Xtr, ytr_orig, Xte, yte_orig):
    pred_tr_log = model.predict(Xtr); pred_te_log = model.predict(Xte)
    pred_tr = np.expm1(pred_tr_log); pred_te = np.expm1(pred_te_log)
    print(f"[{name}] Train MAE={mean_absolute_error(ytr_orig,pred_tr):.4f} RMAE={rmae(ytr_orig,pred_tr):.4f} | "
          f"Test MAE={mean_absolute_error(yte_orig,pred_te):.4f} RMAE={rmae(yte_orig,pred_te):.4f}")

print("\n===== Performance (MAE / RMAE) on original target =====")
for name, mdl in models.items():
    report_model(name, mdl, X_train_fs, y_train_orig, X_test_fs, y_test_orig)

print("\n===== 6-Fold CV on Train (MAE / RMAE) =====")
cv = KFold(n_splits=N_CV, shuffle=True, random_state=RANDOM_STATE)
for name, mdl in models.items():
    mae_scores, rmae_scores = [], []
    for tr, va in cv.split(X_train_fs):
        m = clone(mdl)
        m.fit(X_train_fs.iloc[tr], y_train_log.iloc[tr])
        pred_va = np.expm1(m.predict(X_train_fs.iloc[va]))
        mae_scores.append(mean_absolute_error(y_train_orig.iloc[va], pred_va))
        rmae_scores.append(rmae(y_train_orig.iloc[va], pred_va))
    print(f"[{name}] CV-MAE={np.mean(mae_scores):.4f}  CV-RMAE={np.mean(rmae_scores):.4f}")

# 11) 保存
os.makedirs(MODEL_DIR, exist_ok=True)
bundle = {
    "target_col": target_col,
    "feature_cols": list(X_train_fs.columns),
    "imputer": imp,                 # 带 keep_empty_features=True
    "models": models,               # 训练于 y_log
    "target_transform": "log1p",
    "meta": {"random_state": RANDOM_STATE, "n_cv": N_CV,
             "outlier_rule": "IQR*2.0 OR |Z|<=4.0",
             "feats": ["面积_log1p (if 面积存在)", "area_x_ring (if 面积 & ring_num 存在)"]}
}
joblib.dump(bundle, os.path.join(MODEL_DIR, "linear_models_and_preprocess.joblib"))
print(f"\n[保存] ./models/linear_models_and_preprocess.joblib")


[INFO] split shapes: (83096, 65) (20775, 65)
[INFO] train kept=60072  test=20775
[INFO] final #features: 39
[Tune] Lasso alpha=0.001
[Tune] Ridge alpha=0.001
[Tune] ElasticNet alpha=0.001, l1_ratio=0.5499999999999999

===== Performance (MAE / RMAE) on original target =====
[OLS] Train MAE=759792.0087 RMAE=0.4018 | Test MAE=1053189.0145 RMAE=0.4814
[Lasso] Train MAE=760098.3628 RMAE=0.4020 | Test MAE=1046673.6970 RMAE=0.4799
[Ridge] Train MAE=759792.0071 RMAE=0.4018 | Test MAE=1053188.9829 RMAE=0.4814
[ElasticNet] Train MAE=759879.1504 RMAE=0.4019 | Test MAE=1048650.9052 RMAE=0.4805

===== 6-Fold CV on Train (MAE / RMAE) =====
[OLS] CV-MAE=760430.4879  CV-RMAE=0.4022
[Lasso] CV-MAE=760717.3541  CV-RMAE=0.4024
[Ridge] CV-MAE=760430.4858  CV-RMAE=0.4022
[ElasticNet] CV-MAE=760508.4006  CV-RMAE=0.4023

[保存] ./models/linear_models_and_preprocess.joblib


In [28]:
out = process_real_estate_file(
     input_path="ruc_Class25Q2_test_price.csv",
     output_path="ruc_Class25Q2_test_price__processed.csv",
     city_col="城市", lon_col="lon", lat_col="lat",
     k_per_city={}  # 其他城市默认 k=1
 )
# out.head()

[INFO] Loaded: (34017, 55)  | ID_col=ID
[DROP] 环线
[DROP] 开发商
[DROP] 物业公司
[DROP] 物业办公电话
[SAVE] ruc_Class25Q2_test_price__processed.csv  shape=(34017, 66)


In [29]:
# -*- coding: utf-8 -*-
import pandas as pd
import numpy as np
import joblib

# ========= 路径设置 =========
BUNDLE_PATH = "./models/linear_models_and_preprocess.joblib"
TEST_NUMERIC_PATH = "ruc_Class25Q2_test_price__processed.csv"
OUT_PREFIX = "./preds_"  # 输出文件前缀
# ==========================

# 1) 载入训练打包
bundle = joblib.load(BUNDLE_PATH)
feature_cols = pd.Index(bundle["feature_cols"]).astype(str)  # 模型训练后的真实特征列（如 38 列）
imputer = bundle["imputer"]                                  # 训练阶段的 imputer（可能见过 67 列）
models = bundle["models"]
ttr = bundle.get("target_transform")  # 'log1p' 或 None

print(f"[INFO] bundle loaded. #feature_cols={len(feature_cols)}, models={list(models.keys())}, ttr={ttr}")

# 2) 读取测试数据（数值化后的）
df_new = pd.read_csv(TEST_NUMERIC_PATH)
print(f"[INFO] test csv loaded: {df_new.shape}")

# 2.1 识别并保留 ID
id_col = None
for c in ["ID", "Id", "id"]:
    if c in df_new.columns:
        id_col = c
        break
if id_col is None:
    id_series = pd.Series(np.arange(1, len(df_new)+1), name="ID")
    print("[WARN] 'ID' not found; create sequential ID starting at 1.")
else:
    id_series = df_new[id_col].copy()
    df_new = df_new.drop(columns=[id_col])

# 3) 与训练保持一致的最小特征工程
def fe_minimal(df):
    X = df.copy()
    if "面积" in X.columns:
        X["面积_log1p"] = np.log1p(X["面积"].clip(lower=0))
    if {"面积", "ring_num"}.issubset(X.columns):
        X["area_x_ring"] = X["面积"] * X["ring_num"]
    return X

X_new = fe_minimal(df_new)

# 4) 列名规范化 + 去重
X_new.columns = X_new.columns.astype(str).str.strip()
X_new = X_new.loc[:, ~X_new.columns.duplicated()]

# 5) —— 关键修复：先按 imputer 的列集合对齐，再 impute，最后再选到 feature_cols ——
if hasattr(imputer, "feature_names_in_"):
    imputer_cols = pd.Index(imputer.feature_names_in_).astype(str)  # imputer 拟合时见过的完整列（如 67 列）
else:
    raise RuntimeError("Imputer missing feature_names_in_. Re-train and save imputer with feature names.")

# 5.1 对齐到 imputer_cols（补 NaN + 重排）
missing_to_imp = [c for c in imputer_cols if c not in X_new.columns]
extra_vs_imp = [c for c in X_new.columns if c not in imputer_cols]
print(f"[DEBUG] (align to imputer) missing={len(missing_to_imp)} extra={len(extra_vs_imp)}")
if missing_to_imp:
    print("[DEBUG] missing examples:", missing_to_imp[:30])
if extra_vs_imp:
    print("[DEBUG] extra examples  :", extra_vs_imp[:30])

X_imp_input = X_new.reindex(columns=imputer_cols, fill_value=np.nan)

# 5.2 ndarray 绕过列名检查，做缺失填充
X_imp_np = imputer.transform(X_imp_input.to_numpy())

# 5.3 回到 DataFrame，并“二次选列”到模型真实特征（feature_cols）
X_imp_df = pd.DataFrame(X_imp_np, columns=imputer_cols, index=X_new.index)
X_for_models = X_imp_df.reindex(columns=feature_cols)

# 强校验：此处必须与 feature_cols 等长
assert X_for_models.shape[1] == len(feature_cols), \
    f"Got {X_for_models.shape[1]} features after projection, expected {len(feature_cols)}"

print(f"[INFO] ready for models: shape={X_for_models.shape}")

# 6) 逐模型预测（可传 DataFrame 或 ndarray）
predictions = {}
for name, mdl in models.items():
    yhat = mdl.predict(X_for_models.to_numpy())
    if ttr == "log1p":
        yhat = np.expm1(yhat)
    predictions[name] = yhat
    print(f"[INFO] {name}: example preds = {yhat[:3]}")

# 7) 保存预测（ID + Price）
for name, yhat in predictions.items():
    out_path = f"{OUT_PREFIX}{name}.csv"
    pd.DataFrame({"ID": id_series.values, "Price": yhat}).to_csv(out_path, index=False)
    print(f"[SAVE] {out_path}  rows={len(yhat)}")

print("\n[OK] all predictions saved.")


[INFO] bundle loaded. #feature_cols=39, models=['OLS', 'Lasso', 'Ridge', 'ElasticNet'], ttr=log1p
[INFO] test csv loaded: (34017, 66)
[DEBUG] (align to imputer) missing=0 extra=0
[INFO] ready for models: shape=(34017, 39)
[INFO] OLS: example preds = [26486711.6222387   2474410.28407146  8001714.45232689]
[INFO] Lasso: example preds = [26421230.31194713  2462474.39429353  7961172.64508689]
[INFO] Ridge: example preds = [26486709.17813398  2474410.23842637  8001714.33247743]
[INFO] ElasticNet: example preds = [26383189.23591565  2466903.32838315  7975965.97884499]
[SAVE] ./preds_OLS.csv  rows=34017
[SAVE] ./preds_Lasso.csv  rows=34017
[SAVE] ./preds_Ridge.csv  rows=34017
[SAVE] ./preds_ElasticNet.csv  rows=34017

[OK] all predictions saved.


In [33]:
# -*- coding: utf-8 -*-
import re
import numpy as np
import pandas as pd
from sklearn.model_selection import KFold
from sklearn.cluster import KMeans

# =========================
# —— KMeans 城市中心距离（新增）——
# =========================
_R_EARTH_KM = 6371.0088

def _haversine_km(lon1, lat1, lon2, lat2):
    lon1, lat1, lon2, lat2 = map(np.radians, [lon1, lat1, lon2, lat2])
    dlon, dlat = lon2 - lon1, lat2 - lat1
    a = np.sin(dlat/2.0)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon/2.0)**2
    return 2.0 * _R_EARTH_KM * np.arcsin(np.sqrt(a))

def _kmeans_city_centers_one(df_city, k, lon_col, lat_col, random_state=42):
    """对单个城市的 (lon,lat) 做 KMeans(k)，返回各样本到最近中心的距离（km），索引对齐 df_city.index。"""
    idx = df_city.index
    g = df_city[[lon_col, lat_col]].dropna()
    # 若无有效坐标或样本少于 k，直接返回 NaN
    if g.shape[0] < max(1, k):
        out = pd.Series(np.nan, index=idx, name="dist_to_city_center_km")
        return out

    lat0 = float(g[lat_col].median())
    X = np.c_[ g[lon_col].values * np.cos(np.radians(lat0)), g[lat_col].values ]

    km = KMeans(n_clusters=k, n_init=10, random_state=random_state)
    labels = km.fit_predict(X)
    Cx, Cy = km.cluster_centers_[:, 0], km.cluster_centers_[:, 1]
    # 反变换中心为经纬度
    lon_c = Cx / np.cos(np.radians(lat0))
    lat_c = Cy

    # 到每个中心的球面距离，取最小
    lon_arr, lat_arr = g[lon_col].values, g[lat_col].values
    dmat = np.vstack([
        _haversine_km(lon_arr, lat_arr, lon_c[i], lat_c[i])
        for i in range(k)
    ]).T
    dist_min = dmat.min(axis=1)

    out = pd.Series(np.nan, index=idx, name="dist_to_city_center_km")
    out.loc[g.index] = dist_min
    return out

def _add_city_center_distance_kmeans(df, city_col="城市", lon_col="lon", lat_col="lat", k_per_city=1, random_state=42):
    """
    仅生成一列 dist_to_city_center_km：
    - 对每个城市分别在 (lon,lat) 上做 KMeans(k)，k 可为 int 或 {城市名:k} 的 dict；
    - 计算每条样本到最近中心的 Haversine 距离（km）。
    """
    # 兼容 lat 命名为 'lot' 的情况
    if lat_col not in df.columns and "lot" in df.columns:
        lat_col = "lot"

    if (city_col not in df.columns) or (lon_col not in df.columns) or (lat_col not in df.columns):
        # 缺关键列则直接返回 NaN 列
        return pd.Series(np.nan, index=df.index, name="dist_to_city_center_km")

    # 解析 k_per_city
    if isinstance(k_per_city, int):
        k_default, k_map = k_per_city, {}
    elif isinstance(k_per_city, dict):
        k_default, k_map = 1, k_per_city.copy()
    else:
        raise ValueError("k_per_city 必须是 int 或 dict")

    pieces = []
    for city, g in df.groupby(city_col, sort=False):
        k = k_map.get(city, k_default)
        k = max(1, int(k))
        dist_s = _kmeans_city_centers_one(g, k, lon_col, lat_col, random_state=random_state)
        pieces.append(dist_s)

    dist_all = pd.concat(pieces).loc[df.index]
    dist_all.name = "dist_to_city_center_km"
    return dist_all


# =========================
# —— 基础小工具 —— 
# =========================
def _to_float_first(x):
    """从字符串里抓第一个数字（含千分位/小数）→ float；抓不到返回 NaN"""
    if pd.isna(x): return np.nan
    s = str(x)
    m = re.search(r"(\d{1,3}(?:,\d{3})+|\d+)(?:\.(\d+))?", s)
    if not m: return np.nan
    whole = m.group(1).replace(",", "")
    frac = m.group(2) or ""
    return float(whole + (("." + frac) if frac else ""))

def _extract_all_numbers(s):
    """提取所有数字（含千分位/小数）→ [floats]"""
    if pd.isna(s): return []
    s = str(s)
    out = []
    for m in re.findall(r"(\d{1,3}(?:,\d{3})+|\d+)(?:\.(\d+))?", s):
        whole = m[0].replace(",", "")
        frac = m[1]
        out.append(float(whole + ("." + frac if frac else "")))
    return out

def _parse_layout(x):
    """从‘户型’抽取(室,厅,卫)"""
    if pd.isna(x): 
        return (np.nan, np.nan, np.nan)
    s = str(x)
    room = re.search(r"(\d+)\s*(?:室|房)", s)
    hall = re.search(r"(\d+)\s*(?:厅|客厅)", s)
    bath = re.search(r"(\d+)\s*(?:卫|卫生间|厕)", s)
    r = int(room.group(1)) if room else np.nan
    h = int(hall.group(1)) if hall else np.nan
    b = int(bath.group(1)) if bath else np.nan
    return (r,h,b)

def _process_building_year(year_str):
    """处理‘建筑年代’→(起始年, 结束年)"""
    if pd.isna(year_str) or year_str=='未知': 
        return np.nan, np.nan
    try:
        years = re.findall(r'\d{4}', str(year_str))
        if len(years)>=2: return int(years[0]), int(years[1])
        elif len(years)==1: return int(years[0]), int(years[0])
        else: return np.nan, np.nan
    except:
        return np.nan, np.nan

def _parse_parking_fee(x):
    """停车费：优先抓‘元’前的数，抓不到就取文本所有数字的均值"""
    if pd.isna(x): 
        return np.nan
    s = str(x)
    nums_with_yuan = []
    for m in re.finditer(r"((\d{1,3}(?:,\d{3})+|\d+)(?:\.(\d+))?)\s*元", s):
        num = m.group(1).replace(",", "")
        nums_with_yuan.append(float(num))
    if nums_with_yuan:
        return float(np.mean(nums_with_yuan))
    nums = _extract_all_numbers(s)
    if nums:
        return float(np.mean(nums))
    return np.nan

def _simple_sentiment(text):
    """客户反馈的简单情感：(正计数, 负计数, 情感分)"""
    pos_words = ["满意","干净","安静","方便","舒适","划算","实惠","通透","采光好","靠谱","好评","喜欢","宽敞","安全","交通便利","性价比高","推荐"]
    neg_words = ["差","脏","吵","不方便","贵","坑","闹","潮","霉","不安全","失望","差评","糟糕","噪音","漏水","太小","异味"]
    if pd.isna(text): return (0,0,0.0)
    s = str(text)
    p = sum(1 for w in pos_words if w in s)
    n = sum(1 for w in neg_words if w in s)
    score = (p-n)/(p+n) if (p+n)>0 else 0.0
    return (p,n,score)

def _parse_floor(x):
    """解析楼层→(所在楼层估计, 总楼层)"""
    if pd.isna(x): return (np.nan, np.nan)
    s = str(x)
    m = re.search(r"(\d+)\s*/\s*(\d+)\s*层", s)
    if m:
        return (float(m.group(1)), float(m.group(2)))
    m2 = re.search(r"(低楼层|中楼层|高楼层).*(\d+)\s*层", s)
    if m2:
        lvl = m2.group(1); tot = float(m2.group(2))
        if lvl == "低楼层": fl = tot*0.25
        elif lvl == "中楼层": fl = tot*0.5
        else: fl = tot*0.75
        return (fl, tot)
    m3 = re.search(r"(\d+)\s*层", s)
    if m3:
        return (float(m3.group(1)), np.nan)
    return (np.nan, np.nan)

def _parse_orientation(x):
    """朝向首字编码"""
    if pd.isna(x): return 0
    s = str(x).strip()
    if not s: return 0
    mp = {"东":1,"南":2,"西":3,"北":4}
    return mp.get(s[0], 0)

def _parse_year_only(x):
    """提取 4 位年份为浮点"""
    if pd.isna(x): return np.nan
    s = str(x)
    m = re.search(r"(\d{4})", s)
    return float(m.group(1)) if m else np.nan

def _parse_tenancy_months(x):
    """租期统一到‘月’（区间取均值）"""
    if pd.isna(x): return np.nan
    s = re.sub(r"[至到\-–—]", "~", str(x))
    s = s.replace("个", "")
    m = re.search(r"(\d+(?:\.\d+)?)\s*~\s*(\d+(?:\.\d+)?)\s*(月|年)", s)
    if m:
        a = float(m.group(1)); b = float(m.group(2)); u = m.group(3)
        mid = (a+b)/2.0
        return mid*12 if u=="年" else mid
    m2 = re.search(r"(\d+(?:\.\d+)?)\s*(月|年)", s)
    if m2:
        v = float(m2.group(1)); u = m2.group(2)
        return v*12 if u=="年" else v
    return np.nan


# =========================
# —— 内部：把 DataFrame 转为数值特征（等价于你原来的 make_numeric_features 逻辑）——
# =========================
def _make_numeric_features_core(
    df: pd.DataFrame,
    *,
    add_city_center_dist: bool = True,
    city_col: str = "城市",
    lon_col: str = "lon",
    lat_col: str = "lat",
    k_per_city=1,
    random_state: int = 42,
    drop_cols: list | None = None
) -> pd.DataFrame:
    df = df.copy()
    num_feats = pd.DataFrame(index=df.index)

    # 1) 环线映射
    def map_ring(v):
        if pd.isna(v): return 8
        s = str(v)
        if "内环内" in s: return 1
        if "一至二环" in s: return 2
        if "二至三环" in s: return 3
        if "三至四环" in s: return 4
        if "四至五环" in s: return 5
        if "五至六环" in s: return 6
        if "外环" in s or "六环" in s: return 7
        return 8
    if "环线位置" in df.columns:
        num_feats["ring_num"] = df["环线位置"].apply(map_ring).astype(float)

    # 2) 户型
    if "户型" in df.columns:
        layout = df["户型"].apply(_parse_layout)
        num_feats["室数量"] = layout.apply(lambda t: t[0])
        num_feats["厅数量"] = layout.apply(lambda t: t[1])
        num_feats["卫数量"] = layout.apply(lambda t: t[2])

    # 3) One-Hot
    onehot_cols = ["装修","付款方式","租赁方式","电梯","车位","用水","用电","燃气","采暖","供水","供暖","供电"]
    for col in onehot_cols:
        if col in df.columns:
            dmy = pd.get_dummies(df[col].astype(str), prefix=col, drop_first=True, dummy_na=False)
            num_feats = pd.concat([num_feats, dmy.astype(float)], axis=1)

    # 4) 建筑年代
    if "建筑年代" in df.columns:
        building_years = df["建筑年代"].apply(_process_building_year)
        num_feats["建筑起始年份"] = building_years.apply(lambda x: x[0])
        num_feats["建筑结束年份"] = building_years.apply(lambda x: x[1])
        num_feats["建筑年限"] = 2025 - num_feats["建筑结束年份"]

    # 5) 去单位/提数值
    for c in ["房屋总数","楼栋总数","绿化率","物业费","燃气费","供暖费","面积"]:
        if c in df.columns:
            s = df[c].astype(str)
            if c == "绿化率":
                pct = s.str.extract(r"([\d\.]+)\s*%")[0]
                pct = pd.to_numeric(pct, errors="coerce")
                raw = _to_float_first(s)
                num_feats[c] = np.where(~pct.isna(), pct/100.0, raw)
            else:
                num_feats[c] = s.apply(_to_float_first)

    # 6) 停车费
    if "停车费" in df.columns:
        num_feats["停车费"] = df["停车费"].apply(_parse_parking_fee)

    # 7) 客户反馈情感
    if "客户反馈" in df.columns:
        senti = df["客户反馈"].apply(_simple_sentiment)
        num_feats["feedback_pos_cnt"] = senti.apply(lambda t: t[0]).astype(float)
        num_feats["feedback_neg_cnt"] = senti.apply(lambda t: t[1]).astype(float)
        num_feats["feedback_score"]   = senti.apply(lambda t: t[2]).astype(float)

    # 8) 其余列可数值化的直接保留
    if drop_cols is None:
        drop_cols = ["开发商","物业公司"]
    for c in df.columns:
        if c in num_feats.columns or c in drop_cols:
            continue
        ser = pd.to_numeric(df[c], errors="coerce")
        if ser.notna().any():
            num_feats[c] = ser

    # 9) 楼层
    if "楼层" in df.columns:
        floor_pair = df["楼层"].apply(_parse_floor)
        num_feats["所在楼层"] = floor_pair.apply(lambda t: t[0])
        num_feats["总楼层"] = floor_pair.apply(lambda t: t[1])

    # 10) 朝向编码
    if "朝向" in df.columns:
        num_feats["朝向编码"] = df["朝向"].apply(_parse_orientation).astype(float)

    # 11) 交易时间 -> 年份
    for c in ["交易时间","成交时间","挂牌时间","上架时间"]:
        if c in df.columns and "交易年份" not in num_feats.columns:
            num_feats["交易年份"] = df[c].apply(_parse_year_only)
            break

    # 12) 租期 -> 月
    for c in ["租期","最短租期","最长租期"]:
        if c in df.columns and "租期_月" not in num_feats.columns:
            num_feats["租期_月"] = df[c].apply(_parse_tenancy_months)
            break

    # 13) KMeans 城市中心距离
    if add_city_center_dist:
        dist = _add_city_center_distance_kmeans(
            df, city_col=city_col, lon_col=lon_col, lat_col=lat_col,
            k_per_city=k_per_city, random_state=random_state
        )
        num_feats = pd.concat([num_feats, dist], axis=1)

    # —— 删除经纬度/坐标列（防止 8) 中被保留） ——
    for _col in ["lon", "lat", "coord_x", "coord_y"]:
        if _col in num_feats.columns:
            num_feats.drop(columns=[_col], inplace=True, errors="ignore")

    # 14) 仅保留数值列
    for col in list(num_feats.columns):
        num_feats[col] = pd.to_numeric(num_feats[col], errors="coerce")
    num_feats = num_feats[[c for c in num_feats.columns if str(num_feats[c].dtype) != "object"]]

    return num_feats


# =========================
# —— 主函数：一键处理（集成 KMeans 距离 + 删除 lon/lat）—— 
# =========================
def process_real_estate_file_2(
    input_path: str,
    output_path: str = "processed_numeric.csv",
    # —— KMeans 相关 —— #
    city_col="城市", lon_col="lon", lat_col="lat", k_per_city=1,
    # —— 其余保持模板接口风格（本函数内部主要调用 _make_numeric_features_core） —— #
    heading_print=True
):
    """
    读取 input_path，生成数值特征并新增一列：dist_to_city_center_km（KMeans 城市中心距离，单位：km）。
    同时去除经纬度列，结果仅包含数值特征。
    """
    df = pd.read_csv(input_path)
    if heading_print:
        print(f"[INFO] Loaded: {df.shape}")

    df_out = _make_numeric_features_core(
        df,
        add_city_center_dist=True,
        city_col=city_col, lon_col=lon_col, lat_col=lat_col,
        k_per_city=k_per_city, random_state=42,
        drop_cols=["开发商","物业公司"]
    )

    # 写出
    if output_path:
        df_out.to_csv(output_path, index=False)
        if heading_print:
            print(f"[SAVE] {output_path}  shape={df_out.shape}")

    return df_out




In [36]:
 out = process_real_estate_file_2(
     input_path="ruc_Class25Q2_train_rent.csv",
     output_path="ruc_Class25Q2_train_rent__processed.csv",
     city_col="城市", lon_col="lon", lat_col="lat",
     k_per_city={}  
 )
 out.head()



[INFO] Loaded: (98899, 46)
[SAVE] ruc_Class25Q2_train_rent__processed.csv  shape=(98899, 62)


,ring_num,室数量,厅数量,卫数量,装修_精装修,付款方式_https://img.ljcdn.com/usercent,付款方式_nan,付款方式_半年付价,付款方式_双月付价,付款方式_季付价,...,容 积 率,物业办公电话,停车位,停车费用,所在楼层,总楼层,朝向编码,交易年份,租期_月,dist_to_city_center_km
0,4.0,1.0,1.0,1.0,1.0,0.0,0.0,0.0,0.0,1.0,...,2.5,NaN,450.0,150.0,4.0,6.0,3.0,2024.0,12.0,10.168442
1,3.0,1.0,1.0,1.0,1.0,0.0,0.0,0.0,0.0,1.0,...,1.2,NaN,150.0,150.0,4.0,6.0,2.0,2024.0,NaN,7.656829
2,4.0,1.0,1.0,1.0,1.0,0.0,0.0,0.0,0.0,1.0,...,2.7,NaN,965.0,500.0,1.0,18.0,4.0,2024.0,NaN,6.827118
3,7.0,3.0,1.0,2.0,1.0,0.0,0.0,0.0,0.0,1.0,...,2.8,NaN,500.0,550.0,1.0,10.0,2.0,2024.0,8.5,27.070705
4,4.0,1.0,1.0,1.0,1.0,0.0,0.0,0.0,0.0,1.0,...,1.7,NaN,400.0,150.0,18.0,18.0,2.0,2024.0,12.0,9.229330


In [37]:
# -*- coding: utf-8 -*-
# 目的：y 做 log1p 训练，评估还原到原始尺度；放宽异常值；最小特征工程（面积_log1p、area×ring）
# 修复：Imputer 保留全空列 + 统一填 0；CV 使用 sklearn.base.clone

import os, re, warnings, numpy as np, pandas as pd, joblib
from sklearn.model_selection import train_test_split, GridSearchCV, KFold
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression, Lasso, Ridge, ElasticNet
from sklearn.metrics import mean_absolute_error
from sklearn.base import clone

warnings.filterwarnings("ignore")

DATA_PATH = "ruc_Class25Q2_train_rent__processed.csv"
RANDOM_STATE = 111
TEST_SIZE = 0.2
N_CV = 6
MODEL_DIR = "./models"

TARGET_KEYS = ["Price","price","租金","总价","成交价","月租","租价","y","target","label"]
LEAK_PATTERNS = [r"价格", r"单价", r"均价", r"总价", r"成交价", r"租金", r"租价", r"Price", r"price", r"每平米"]

def rmae(y_true, y_pred, eps=1e-8):
    y_true = np.asarray(y_true); y_pred = np.asarray(y_pred)
    return np.mean(np.abs(y_pred - y_true) / np.maximum(np.abs(y_true), eps))

# 1) 读取 + 目标列识别
df = pd.read_csv(DATA_PATH)
target_col = None
for k in TARGET_KEYS:
    if k in df.columns and pd.to_numeric(df[k], errors="coerce").notna().mean() >= 0.5:
        target_col = k; break
if target_col is None:
    numeric_cols = [c for c in df.columns if pd.api.types.is_numeric_dtype(df[c])]
    target_col = max(numeric_cols, key=lambda c: df[c].notna().sum())

y_full = pd.to_numeric(df[target_col], errors="coerce")
X_full = df.drop(columns=[target_col])

# 2) 去泄露（按列名）+ 仅数值列
def is_leaky(colname): return any(re.search(p, colname, flags=re.I) for p in LEAK_PATTERNS)
leak_cols = [c for c in X_full.columns if is_leaky(c)]
if leak_cols:
    print("[WARN] drop leakage-like columns:", leak_cols)
    X_full = X_full.drop(columns=leak_cols)
X_full = X_full[[c for c in X_full.columns if pd.api.types.is_numeric_dtype(X_full[c])]]

# 3) 划分（80/20）
mask = y_full.notna()
X = X_full.loc[mask].copy(); y = y_full.loc[mask].copy()
X_train, X_test, y_train_orig, y_test_orig = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE
)

print("[INFO] split shapes:", X_train.shape, X_test.shape)

# 4) 缺失填充（仅训练集统计）——关键修复：保留全空列，并在之后把全空列置 0
imp = SimpleImputer(strategy="median", keep_empty_features=True)
X_train_imp_np = imp.fit_transform(X_train)
X_test_imp_np  = imp.transform(X_test)

X_train_imp = pd.DataFrame(X_train_imp_np, columns=X_train.columns, index=X_train.index)
X_test_imp  = pd.DataFrame(X_test_imp_np,  columns=X_test.columns,  index=X_test.index)

empty_cols = [c for c in X_train.columns if X_train[c].isna().all()]
if empty_cols:
    print("[WARN] all-NaN columns filled with 0:", empty_cols[:20], "..." if len(empty_cols)>20 else "")
    X_train_imp[empty_cols] = X_train_imp[empty_cols].fillna(0.0)
    X_test_imp[empty_cols]  = X_test_imp[empty_cols].fillna(0.0)

# 5) 放宽异常值规则（仅训练集）
Q1, Q3 = X_train_imp.quantile(0.25), X_train_imp.quantile(0.75)
IQR = Q3 - Q1
lower = Q1 - 2.0 * IQR
upper = Q3 + 2.0 * IQR
keep_iqr_relaxed = ((X_train_imp >= lower) & (X_train_imp <= upper)).all(axis=1)

mu = X_train_imp.mean()
sd = X_train_imp.std(ddof=0).replace(0, 1.0)
Z = (X_train_imp - mu) / sd
keep_z_relaxed = (Z.abs() <= 4.0).all(axis=1)

keep_mask = keep_iqr_relaxed | keep_z_relaxed
X_train_clean = X_train_imp.loc[keep_mask].copy()
y_train_orig = y_train_orig.loc[keep_mask].copy()

print(f"[INFO] train kept={len(X_train_clean)}  test={len(X_test_imp)}")

# 6) 最小特征工程：面积_log1p、area×ring
def add_minimal_fe(Xtr: pd.DataFrame, Xte: pd.DataFrame):
    Xtr2, Xte2 = Xtr.copy(), Xte.copy()
    if "面积" in Xtr2.columns:
        Xtr2["面积_log1p"] = np.log1p(Xtr2["面积"].clip(lower=0))
        Xte2["面积_log1p"] = np.log1p(Xte2["面积"].clip(lower=0))
    if {"面积","ring_num"}.issubset(Xtr2.columns):
        Xtr2["area_x_ring"] = Xtr2["面积"] * Xtr2["ring_num"]
        Xte2["area_x_ring"] = Xte2["面积"] * Xte2["ring_num"]
    return Xtr2, Xte2

X_train_fe, X_test_fe = add_minimal_fe(X_train_clean, X_test_imp)

# 7) y 做 log1p 变换（训练域）
y_train_log = np.log1p(y_train_orig.clip(lower=0))
y_test_log  = np.log1p(y_test_orig.clip(lower=0))  # 仅用于观察

# 8) 轻量特征选择（去弱相关、去高相关、简易VIF）
corr = pd.DataFrame({"corr": X_train_fe.corrwith(y_train_log)})
keep_corr = corr.index[(corr["corr"].abs() >= 0.02)].tolist()
if len(keep_corr) == 0:
    # 兜底：保留前 20 个方差最大的列，避免空特征集
    keep_corr = X_train_fe.var().sort_values(ascending=False).index[:20].tolist()

X_train_fs = X_train_fe[keep_corr].copy(); X_test_fs = X_test_fe[keep_corr].copy()

corr_mat = X_train_fs.corr().abs()
upper = corr_mat.where(np.triu(np.ones(corr_mat.shape), k=1).astype(bool))
high_cols = [c for c in upper.columns if any(upper[c] > 0.8)]
if high_cols:
    X_train_fs.drop(columns=high_cols, inplace=True, errors="ignore")
    X_test_fs.drop(columns=high_cols, inplace=True, errors="ignore")

def compute_vif(Xm: pd.DataFrame):
    from sklearn.linear_model import LinearRegression
    vifs = {}
    for col in Xm.columns:
        y_i = Xm[col].values
        X_i = Xm.drop(columns=[col]).values
        if X_i.shape[1]==0: vifs[col]=1.0; continue
        r2 = LinearRegression().fit(X_i, y_i).score(X_i, y_i)
        vifs[col] = np.inf if r2>=0.9999 else 1/(1-r2)
    return pd.Series(vifs)

while True:
    if X_train_fs.shape[1] <= 5: break
    vif = compute_vif(X_train_fs.fillna(0))
    mx = vif.idxmax(); mv = float(vif.max())
    if mv > 10 and len(X_train_fs.columns) > 5:
        X_train_fs.drop(columns=[mx], inplace=True); X_test_fs.drop(columns=[mx], inplace=True)
    else:
        break

print("[INFO] final #features:", X_train_fs.shape[1])

# 9) 四个线性模型（在 y_log 上训练），并调参
models = {}

pipe_ols = Pipeline([("sc", StandardScaler()), ("lr", LinearRegression())]).fit(X_train_fs, y_train_log)
models["OLS"] = pipe_ols

pipe_lasso = Pipeline([("sc", StandardScaler()), ("ls", Lasso(max_iter=10000, random_state=RANDOM_STATE))])
gs_lasso = GridSearchCV(pipe_lasso, {"ls__alpha": np.logspace(-3, 1, 12)}, cv=N_CV,
                        scoring="neg_mean_absolute_error", n_jobs=-1)
gs_lasso.fit(X_train_fs, y_train_log); models["Lasso"] = gs_lasso.best_estimator_
print(f"[Tune] Lasso alpha={gs_lasso.best_params_['ls__alpha']}")

pipe_ridge = Pipeline([("sc", StandardScaler()), ("rg", Ridge(random_state=RANDOM_STATE))])
gs_ridge = GridSearchCV(pipe_ridge, {"rg__alpha": np.logspace(-3, 3, 13)}, cv=N_CV,
                        scoring="neg_mean_absolute_error", n_jobs=-1)
gs_ridge.fit(X_train_fs, y_train_log); models["Ridge"] = gs_ridge.best_estimator_
print(f"[Tune] Ridge alpha={gs_ridge.best_params_['rg__alpha']}")

pipe_en = Pipeline([("sc", StandardScaler()), ("en", ElasticNet(max_iter=10000, random_state=RANDOM_STATE))])
param_grid_en = {"en__alpha": np.logspace(-3, 1, 8), "en__l1_ratio": np.linspace(0.05, 0.95, 10)}
gs_en = GridSearchCV(pipe_en, param_grid_en, cv=N_CV,
                     scoring="neg_mean_absolute_error", n_jobs=-1)
gs_en.fit(X_train_fs, y_train_log); models["ElasticNet"] = gs_en.best_estimator_
print(f"[Tune] ElasticNet alpha={gs_en.best_params_['en__alpha']}, l1_ratio={gs_en.best_params_['en__l1_ratio']}")

# 10) 评估（还原到原始 y 空间）
def report_model(name, model, Xtr, ytr_orig, Xte, yte_orig):
    pred_tr_log = model.predict(Xtr); pred_te_log = model.predict(Xte)
    pred_tr = np.expm1(pred_tr_log); pred_te = np.expm1(pred_te_log)
    print(f"[{name}] Train MAE={mean_absolute_error(ytr_orig,pred_tr):.4f} RMAE={rmae(ytr_orig,pred_tr):.4f} | "
          f"Test MAE={mean_absolute_error(yte_orig,pred_te):.4f} RMAE={rmae(yte_orig,pred_te):.4f}")

print("\n===== Performance (MAE / RMAE) on original target =====")
for name, mdl in models.items():
    report_model(name, mdl, X_train_fs, y_train_orig, X_test_fs, y_test_orig)

print("\n===== 6-Fold CV on Train (MAE / RMAE) =====")
cv = KFold(n_splits=N_CV, shuffle=True, random_state=RANDOM_STATE)
for name, mdl in models.items():
    mae_scores, rmae_scores = [], []
    for tr, va in cv.split(X_train_fs):
        m = clone(mdl)
        m.fit(X_train_fs.iloc[tr], y_train_log.iloc[tr])
        pred_va = np.expm1(m.predict(X_train_fs.iloc[va]))
        mae_scores.append(mean_absolute_error(y_train_orig.iloc[va], pred_va))
        rmae_scores.append(rmae(y_train_orig.iloc[va], pred_va))
    print(f"[{name}] CV-MAE={np.mean(mae_scores):.4f}  CV-RMAE={np.mean(rmae_scores):.4f}")

# 11) 保存
os.makedirs(MODEL_DIR, exist_ok=True)
bundle = {
    "target_col": target_col,
    "feature_cols": list(X_train_fs.columns),
    "imputer": imp,                 # 带 keep_empty_features=True
    "models": models,               # 训练于 y_log
    "target_transform": "log1p",
    "meta": {"random_state": RANDOM_STATE, "n_cv": N_CV,
             "outlier_rule": "IQR*2.0 OR |Z|<=4.0",
             "feats": ["面积_log1p (if 面积存在)", "area_x_ring (if 面积 & ring_num 存在)"]}
}
joblib.dump(bundle, os.path.join(MODEL_DIR, "linear_models_and_preprocess.joblib"))
print(f"\n[保存] ./models/linear_models_and_preprocess.joblib")


[INFO] split shapes: (79119, 61) (19780, 61)
[INFO] train kept=54894  test=19780
[INFO] final #features: 35
[Tune] Lasso alpha=0.001
[Tune] Ridge alpha=0.001
[Tune] ElasticNet alpha=0.001, l1_ratio=0.05

===== Performance (MAE / RMAE) on original target =====
[OLS] Train MAE=166998.2743 RMAE=0.3492 | Test MAE=197370.5716 RMAE=0.3912
[Lasso] Train MAE=167283.3041 RMAE=0.3494 | Test MAE=197256.4194 RMAE=0.3910
[Ridge] Train MAE=166998.2752 RMAE=0.3492 | Test MAE=197370.5717 RMAE=0.3912
[ElasticNet] Train MAE=167058.7163 RMAE=0.3492 | Test MAE=197370.9669 RMAE=0.3911

===== 6-Fold CV on Train (MAE / RMAE) =====
[OLS] CV-MAE=167115.6983  CV-RMAE=0.3495
[Lasso] CV-MAE=167398.5017  CV-RMAE=0.3497
[Ridge] CV-MAE=167115.6993  CV-RMAE=0.3495
[ElasticNet] CV-MAE=167175.7396  CV-RMAE=0.3495

[保存] ./models/linear_models_and_preprocess.joblib


In [38]:
 out = process_real_estate_file_2(
     input_path="ruc_Class25Q2_test_rent.csv",
     output_path="ruc_Class25Q2_test_rent__processed.csv",
     city_col="城市", lon_col="lon", lat_col="lat",
     k_per_city={}  
 )
 out.head()



[INFO] Loaded: (9773, 46)
[SAVE] ruc_Class25Q2_test_rent__processed.csv  shape=(9773, 59)


,ring_num,室数量,厅数量,卫数量,装修_精装修,付款方式_半年付价,付款方式_双月付价,付款方式_季付价,付款方式_年付价,付款方式_月付价,...,容 积 率,物业办公电话,停车位,停车费用,所在楼层,总楼层,朝向编码,交易年份,租期_月,dist_to_city_center_km
0,8.0,2.0,2.0,1,1.0,0.0,0.0,0.0,0.0,0.0,...,2.5,NaN,1600.0,100.0,2.0,8.0,2.0,2025.0,NaN,44.568083
1,8.0,2.0,1.0,1,1.0,0.0,0.0,0.0,0.0,1.0,...,3.0,NaN,200.0,800.0,2.0,8.0,2.0,2025.0,NaN,17.961806
2,8.0,2.0,2.0,1,1.0,0.0,0.0,0.0,0.0,0.0,...,NaN,NaN,NaN,NaN,0.0,0.0,2.0,2025.0,12.0,30.278488
3,4.0,2.0,1.0,1,1.0,0.0,0.0,1.0,0.0,0.0,...,2.9,NaN,2317.0,1200.0,1.5,2.0,1.0,2025.0,NaN,12.516696
4,8.0,3.0,2.0,2,1.0,0.0,0.0,1.0,0.0,0.0,...,NaN,NaN,NaN,NaN,1.5,3.0,2.0,2025.0,NaN,19.537780


In [39]:
# -*- coding: utf-8 -*-
import pandas as pd
import numpy as np
import joblib

# ========= 路径设置 =========
BUNDLE_PATH = "./models/linear_models_and_preprocess.joblib"
TEST_NUMERIC_PATH = "ruc_Class25Q2_test_rent__processed.csv"
OUT_PREFIX = "./preds2_"  # 输出文件前缀
# ==========================

# 1) 载入训练打包
bundle = joblib.load(BUNDLE_PATH)
feature_cols = pd.Index(bundle["feature_cols"]).astype(str)  
imputer = bundle["imputer"]                                  
models = bundle["models"]
ttr = bundle.get("target_transform")  # 'log1p' 或 None

print(f"[INFO] bundle loaded. #feature_cols={len(feature_cols)}, models={list(models.keys())}, ttr={ttr}")

# 2) 读取测试数据（数值化后的）
df_new = pd.read_csv(TEST_NUMERIC_PATH)
print(f"[INFO] test csv loaded: {df_new.shape}")

# 2.1 识别并保留 ID
id_col = None
for c in ["ID", "Id", "id"]:
    if c in df_new.columns:
        id_col = c
        break
if id_col is None:
    id_series = pd.Series(np.arange(1, len(df_new)+1), name="ID")
    print("[WARN] 'ID' not found; create sequential ID starting at 1.")
else:
    id_series = df_new[id_col].copy()
    df_new = df_new.drop(columns=[id_col])

# 3) 与训练保持一致的最小特征工程
def fe_minimal(df):
    X = df.copy()
    if "面积" in X.columns:
        X["面积_log1p"] = np.log1p(X["面积"].clip(lower=0))
    if {"面积", "ring_num"}.issubset(X.columns):
        X["area_x_ring"] = X["面积"] * X["ring_num"]
    return X

X_new = fe_minimal(df_new)

# 4) 列名规范化 + 去重
X_new.columns = X_new.columns.astype(str).str.strip()
X_new = X_new.loc[:, ~X_new.columns.duplicated()]

# 5) —— 关键修复：先按 imputer 的列集合对齐，再 impute，最后再选到 feature_cols ——
if hasattr(imputer, "feature_names_in_"):
    imputer_cols = pd.Index(imputer.feature_names_in_).astype(str)  # imputer 拟合时见过的完整列（如 67 列）
else:
    raise RuntimeError("Imputer missing feature_names_in_. Re-train and save imputer with feature names.")

# 5.1 对齐到 imputer_cols（补 NaN + 重排）
missing_to_imp = [c for c in imputer_cols if c not in X_new.columns]
extra_vs_imp = [c for c in X_new.columns if c not in imputer_cols]
print(f"[DEBUG] (align to imputer) missing={len(missing_to_imp)} extra={len(extra_vs_imp)}")
if missing_to_imp:
    print("[DEBUG] missing examples:", missing_to_imp[:30])
if extra_vs_imp:
    print("[DEBUG] extra examples  :", extra_vs_imp[:30])

X_imp_input = X_new.reindex(columns=imputer_cols, fill_value=np.nan)

# 5.2 ndarray 绕过列名检查，做缺失填充
X_imp_np = imputer.transform(X_imp_input.to_numpy())

# 5.3 回到 DataFrame，并“二次选列”到模型真实特征（feature_cols）
X_imp_df = pd.DataFrame(X_imp_np, columns=imputer_cols, index=X_new.index)
X_for_models = X_imp_df.reindex(columns=feature_cols)

# 强校验：此处必须与 feature_cols 等长
assert X_for_models.shape[1] == len(feature_cols), \
    f"Got {X_for_models.shape[1]} features after projection, expected {len(feature_cols)}"

print(f"[INFO] ready for models: shape={X_for_models.shape}")

# 6) 逐模型预测（可传 DataFrame 或 ndarray）
predictions = {}
for name, mdl in models.items():
    yhat = mdl.predict(X_for_models.to_numpy())
    if ttr == "log1p":
        yhat = np.expm1(yhat)
    predictions[name] = yhat
    print(f"[INFO] {name}: example preds = {yhat[:3]}")

# 7) 保存预测（ID + Price）
for name, yhat in predictions.items():
    out_path = f"{OUT_PREFIX}{name}.csv"
    pd.DataFrame({"ID": id_series.values, "Price": yhat}).to_csv(out_path, index=False)
    print(f"[SAVE] {out_path}  rows={len(yhat)}")

print("\n[OK] all predictions saved.")


[INFO] bundle loaded. #feature_cols=35, models=['OLS', 'Lasso', 'Ridge', 'ElasticNet'], ttr=log1p
[INFO] test csv loaded: (9773, 59)
[DEBUG] (align to imputer) missing=3 extra=2
[DEBUG] missing examples: ['付款方式_https://img.ljcdn.com/usercent', '付款方式_nan', '电梯_无']
[DEBUG] extra examples  : ['面积_log1p', 'area_x_ring']
[INFO] ready for models: shape=(9773, 35)
[INFO] OLS: example preds = [250223.29314581 521743.91302963 366578.37130669]
[INFO] Lasso: example preds = [251007.19522756 525444.40398282 365461.13197966]
[INFO] Ridge: example preds = [250223.29628657 521743.921332   366578.37123529]
[INFO] ElasticNet: example preds = [250432.85170348 522374.04761347 366520.40196661]
[SAVE] ./preds2_OLS.csv  rows=9773
[SAVE] ./preds2_Lasso.csv  rows=9773
[SAVE] ./preds2_Ridge.csv  rows=9773
[SAVE] ./preds2_ElasticNet.csv  rows=9773

[OK] all predictions saved.
